# 08 — Leakage-safe richer snapshot features and ordinary classifiers

## Goal
Build richer **one-row-per-student** features at fixed signup-relative cutoffs (7, 14, 30 days), then compare ordinary classifiers using the project objective: **maximise validation recall subject to a precision floor**.

## Feature safety contract
- Each row uses quiz attempts with `signup_dt <= attempt_dt <= signup_dt + horizon` only.
- No last-observed-attempt, future attempt count, refund status, explanation text/labels, IDs, or post-cutoff outcome-derived features are predictors.
- Static student attributes are treated as available at signup. Verify this assumption before production use.
- Imputation, one-hot encoding, scaling, and fitting happen inside pipelines on training data only.
- One shared student-level chronological split is used for all horizons.

## Important limitation
The outcome has no dropout timestamp. This predicts the supplied student-level label, not a verified dropout event after the cutoff. Also, the dataset is small; treat model comparisons as exploratory.

In [1]:
from pathlib import Path
import warnings, json, pickle
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score, roc_auc_score, confusion_matrix

ROOT_CANDIDATES = [Path("."), Path(".."), Path("/mnt/data/student_success_assistant")]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "data" / "students.csv").exists()), Path.cwd())
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "rich_snapshot"
FEATURES_OUT = OUT / "features"; METRICS_OUT = OUT / "metrics"; PRED_OUT = OUT / "predictions"; MODELS_OUT = OUT / "models"
for p in [FEATURES_OUT, METRICS_OUT, PRED_OUT, MODELS_OUT]: p.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
PRECISION_FLOORS = [0.40, 0.50, 0.60]
print('Project root:', ROOT)

Project root: /Users/abishek/Documents/Projects/student_success_assistant


## 1. Load and validate source tables

In [2]:
students = pd.read_csv(DATA / "students.csv")
attempts = pd.read_csv(DATA / "quiz_attempts.csv")
outcomes = pd.read_csv(DATA / "outcome.csv")

students["signup_dt"] = pd.to_datetime(students["signup_date"], errors="coerce")
attempts["attempt_dt"] = pd.to_datetime(attempts["attempted_at"], errors="coerce")
attempts["score_num"] = pd.to_numeric(attempts["score"], errors="coerce")
attempts["duration_num"] = pd.to_numeric(attempts["time_spent_min"], errors="coerce")
attempts["score_valid"] = attempts["score_num"].between(0, 100)
attempts["score_invalid_flag"] = (attempts["score_num"].notna() & ~attempts["score_valid"]).astype(int)
attempts["duration_valid"] = attempts["duration_num"].ge(0)
outcomes["target_dropout"] = outcomes["dropped_out"].astype(str).str.strip().str.lower().map({"yes":1,"no":0})

assert students.student_id.is_unique and students.student_id.notna().all()
assert outcomes.student_id.is_unique and outcomes.target_dropout.notna().all()
assert attempts.attempt_id.is_unique and attempts.attempt_id.notna().all()
assert set(attempts.student_id.dropna()).issubset(set(students.student_id))
assert set(outcomes.student_id).issubset(set(students.student_id))
student_table = students.merge(outcomes[["student_id","target_dropout"]], on="student_id", how="inner", validate="one_to_one")
attempts = attempts.merge(students[["student_id","signup_dt"]], on="student_id", how="left", validate="many_to_one")
observation_end = attempts.attempt_dt.max()
print('Students:',len(student_table),'attempts:',len(attempts),'observation end:',observation_end)
print('Label prevalence:',student_table.target_dropout.mean())

Students: 500 attempts: 2453 observation end: 2026-09-28 14:00:00
Label prevalence: 0.25


## 2. Build richer features from events available by the cutoff

Added feature groups:
- activity counts in recent windows (last 3/7/14 days relative to cutoff);
- score level, dispersion, first-to-last change, recent-vs-earlier change, last-three mean;
- activity regularity (mean/maximum gap, gap variability, days since last attempt at cutoff);
- lesson coverage and per-lesson score aggregates;
- duration quality and duration aggregates;
- missing/invalid score indicators.

Every event-derived feature is calculated after filtering attempts to the student's signup-to-cutoff interval. The features `days_since_last_attempt_at_cutoff` and all recent-window features are based only on events observed by that cutoff.

In [3]:
def safe_slope(vals, times):
    if len(vals) < 2: return np.nan
    x = (pd.to_datetime(times) - pd.to_datetime(times).min()).dt.total_seconds().to_numpy()/86400.0
    y = np.asarray(vals, dtype=float)
    if len(np.unique(x)) < 2: return np.nan
    return float(np.polyfit(x, y, 1)[0])

def build_row(s, all_attempts, horizon, target):
    signup = s.signup_dt
    cutoff = signup + pd.Timedelta(days=horizon) if pd.notna(signup) else pd.NaT
    d = all_attempts[all_attempts.student_id.eq(s.student_id)].copy()
    d = d[d.attempt_dt.notna() & pd.notna(signup) & (d.attempt_dt >= signup) & (d.attempt_dt <= cutoff)].sort_values(["attempt_dt","attempt_id"])
    valid = d[d.score_valid & d.score_num.notna()].copy()
    durations = d.loc[d.duration_valid & d.duration_num.notna(),"duration_num"].astype(float)
    row = {
        "student_id":s.student_id, "snapshot_days":int(horizon), "snapshot_cutoff":cutoff,
        "max_source_event_dt":d.attempt_dt.max() if len(d) else pd.NaT,
        "target_dropout":int(target),
        "age_group":getattr(s,"age_group",np.nan), "country":getattr(s,"country",np.nan), "device":getattr(s,"device",np.nan),
        "signup_weekday":signup.dayofweek if pd.notna(signup) else np.nan,
        "signup_month":signup.month if pd.notna(signup) else np.nan,
        "days_to_first_attempt":((d.attempt_dt.min()-signup).total_seconds()/86400 if len(d) and pd.notna(signup) else np.nan),
        "attempt_count":len(d), "valid_score_count":len(valid),
        "invalid_score_count":int(d.score_invalid_flag.sum()), "score_missing_count":int(d.score_num.isna().sum()),
        "duration_observed_count":len(durations), "duration_missing_count":int(d.duration_num.isna().sum()),
        "duration_negative_count":int((d.duration_num<0).fillna(False).sum()),
        "distinct_lessons_attempted":d.lesson_id.nunique(dropna=True),
        "active_days":d.attempt_dt.dt.normalize().nunique(),
        "mean_score":valid.score_num.mean(), "median_score":valid.score_num.median(),
        "std_score":valid.score_num.std(ddof=1) if len(valid)>1 else np.nan,
        "min_score":valid.score_num.min(), "max_score":valid.score_num.max(),
        "score_range":(valid.score_num.max()-valid.score_num.min()) if len(valid) else np.nan,
        "latest_valid_score":valid.score_num.iloc[-1] if len(valid) else np.nan,
        "first_valid_score":valid.score_num.iloc[0] if len(valid) else np.nan,
        "score_change_first_to_last":(valid.score_num.iloc[-1]-valid.score_num.iloc[0]) if len(valid)>1 else np.nan,
        "last3_mean_score":valid.score_num.tail(3).mean(),
        "last3_score_std":valid.score_num.tail(3).std(ddof=1) if len(valid.tail(3))>1 else np.nan,
        "score_slope_points_per_day":safe_slope(valid.score_num.to_list(),valid.attempt_dt) if len(valid)>1 else np.nan,
        "mean_duration_min":durations.mean(), "median_duration_min":durations.median(),
        "no_attempts_flag":int(len(d)==0), "no_valid_scores_flag":int(len(valid)==0),
    }
    if len(valid)>=4:
        row["recent3_minus_earlier_mean"] = float(valid.score_num.tail(3).mean()-valid.score_num.iloc[:-3].mean())
    else: row["recent3_minus_earlier_mean"] = np.nan
    if len(d)>1:
        gaps = d.attempt_dt.diff().dt.total_seconds().dropna()/86400.0
        row["mean_gap_days"] = gaps.mean(); row["max_gap_days"] = gaps.max(); row["std_gap_days"] = gaps.std(ddof=1) if len(gaps)>1 else np.nan
    else:
        row["mean_gap_days"] = np.nan; row["max_gap_days"] = np.nan; row["std_gap_days"] = np.nan
    row["days_since_last_attempt_at_cutoff"] = ((cutoff-d.attempt_dt.max()).total_seconds()/86400.0 if len(d) and pd.notna(cutoff) else np.nan)
    for window in [3,7,14]:
        w = d[d.attempt_dt >= cutoff-pd.Timedelta(days=window)] if pd.notna(cutoff) else d.iloc[0:0]
        wv = w[w.score_valid & w.score_num.notna()]
        row[f"attempts_last_{window}d"] = len(w)
        row[f"active_days_last_{window}d"] = w.attempt_dt.dt.normalize().nunique()
        row[f"mean_score_last_{window}d"] = wv.score_num.mean()
        row[f"valid_scores_last_{window}d"] = len(wv)
    # Per-lesson aggregates based only on eligible attempts. Use aggregate numeric features, not IDs as categories.
    lesson_counts = d.groupby("lesson_id",dropna=True).size()
    row["max_attempts_single_lesson"] = lesson_counts.max() if len(lesson_counts) else 0
    row["lessons_with_multiple_attempts"] = int((lesson_counts>1).sum()) if len(lesson_counts) else 0
    lesson_scores = valid.groupby("lesson_id",dropna=True).score_num.mean()
    row["between_lesson_score_std"] = lesson_scores.std(ddof=1) if len(lesson_scores)>1 else np.nan
    row["lowest_lesson_mean_score"] = lesson_scores.min() if len(lesson_scores) else np.nan
    row["highest_lesson_mean_score"] = lesson_scores.max() if len(lesson_scores) else np.nan
    return row

rows=[]
for s in student_table.itertuples(index=False):
    for h in [7,14,30]:
        rows.append(build_row(s,attempts,h,s.target_dropout))
rich = pd.DataFrame(rows)
for h in [7,14,30]:
    part=rich[rich.snapshot_days.eq(h)].copy()
    assert part.student_id.is_unique
    # Audit that every included event timestamp is at/before its snapshot cutoff.
    part["max_source_event_dt"] = pd.to_datetime(part["max_source_event_dt"], errors="coerce")
    part["snapshot_cutoff"] = pd.to_datetime(part["snapshot_cutoff"], errors="coerce")
    assert (part["max_source_event_dt"].isna() | (part["max_source_event_dt"] <= part["snapshot_cutoff"])).all()
    part.to_csv(FEATURES_OUT/f"rich_snapshot_{h}d.csv",index=False)
    print(f'{h}d:',part.shape,'positives:',int(part.target_dropout.sum()))
display(rich.head())

7d: (500, 58) positives: 125
14d: (500, 58) positives: 125
30d: (500, 58) positives: 125


,student_id,snapshot_days,snapshot_cutoff,max_source_event_dt,target_dropout,age_group,country,device,signup_weekday,signup_month,...,valid_scores_last_7d,attempts_last_14d,active_days_last_14d,mean_score_last_14d,valid_scores_last_14d,max_attempts_single_lesson,lessons_with_multiple_attempts,between_lesson_score_std,lowest_lesson_mean_score,highest_lesson_mean_score
0,S001,7,2026-07-25,2026-07-22 14:01:00,0,35-44,UAE,mobile,5,7,...,2,2,2,89.000000,2,2,1,NaN,89.000000,89.000000
1,S001,14,2026-08-01,2026-07-27 20:16:00,0,35-44,UAE,mobile,5,7,...,1,3,3,86.666667,3,3,1,NaN,86.666667,86.666667
2,S001,30,2026-08-17,2026-07-27 20:16:00,0,35-44,UAE,mobile,5,7,...,0,0,0,NaN,0,3,1,NaN,86.666667,86.666667
3,S002,7,2026-07-18,2026-07-17 01:33:00,0,18-24,Sri Lanka,mobile,5,7,...,1,1,1,77.000000,1,1,0,NaN,77.000000,77.000000
4,S002,14,2026-07-25,2026-07-17 01:33:00,0,18-24,Sri Lanka,mobile,5,7,...,0,1,1,77.000000,1,1,0,NaN,77.000000,77.000000


## 3. Shared chronological student split

Split by signup cohort, not by attempt rows. The split is fixed once and reused for all horizons. With a small dataset, this split may have few positives; the notebook reports the class counts so infeasible validation thresholds are explicit.

In [4]:
# One row per student to create shared split boundaries by signup date.
student_order = student_table[["student_id","signup_dt","target_dropout"]].sort_values(["signup_dt","student_id"]).reset_index(drop=True)
n=len(student_order); n_train=int(n*0.60); n_val=int(n*0.20)
train_ids=set(student_order.iloc[:n_train].student_id)
val_ids=set(student_order.iloc[n_train:n_train+n_val].student_id)
test_ids=set(student_order.iloc[n_train+n_val:].student_id)
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)
split_map={sid:('train' if sid in train_ids else 'validation' if sid in val_ids else 'test') for sid in student_order.student_id}
split_report=[]
for split_name, ids in [('train',train_ids),('validation',val_ids),('test',test_ids)]:
    y=student_order[student_order.student_id.isin(ids)].target_dropout
    split_report.append({'split':split_name,'students':len(ids),'positives':int(y.sum()),'prevalence':float(y.mean()),'min_signup':str(student_order[student_order.student_id.isin(ids)].signup_dt.min()),'max_signup':str(student_order[student_order.student_id.isin(ids)].signup_dt.max())})
split_report=pd.DataFrame(split_report); display(split_report)
split_report.to_csv(METRICS_OUT/'shared_chronological_split.csv',index=False)
with open(OUT/'split_ids.json','w') as f: json.dump({k:sorted(v) for k,v in {'train':train_ids,'validation':val_ids,'test':test_ids}.items()},f,indent=2)

,split,students,positives,prevalence,min_signup,max_signup
0,train,300,67,0.223333,2026-06-01 00:00:00,2026-07-07 00:00:00
1,validation,100,27,0.270000,2026-07-07 00:00:00,2026-07-19 00:00:00
2,test,100,31,0.310000,2026-07-19 00:00:00,2026-07-31 00:00:00


## 4. Train ordinary classifiers and select thresholds on validation only

Models: logistic regression, random forest, and histogram gradient boosting. One-hot encoding is dense for histogram boosting; the sample is small enough for this. If your local scikit-learn version lacks `sparse_output`, change it to `sparse=False`. No test-set threshold tuning is performed.

In [5]:
def choose_threshold(y, p, floor):
    candidates=np.unique(np.r_[0.0,p,1.0])
    feasible=[]
    for t in candidates:
        pred=(p>=t).astype(int)
        prec=precision_score(y,pred,zero_division=0)
        rec=recall_score(y,pred,zero_division=0)
        if prec>=floor: feasible.append((rec,prec,t))
    if not feasible: return np.nan,False,np.nan,np.nan
    rec,prec,t=max(feasible,key=lambda x:(x[0],x[1],x[2]))
    return float(t),True,float(rec),float(prec)

def metric_row(y,p,t):
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
            'TP':int(tp),'FP':int(fp),'FN':int(fn),'TN':int(tn),'flagged':int(pred.sum()),
            'flagged_rate':float(pred.mean()),'PR_AUC':average_precision_score(y,p),
            'ROC_AUC':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan}

EXCLUDE={'student_id','snapshot_days','snapshot_cutoff','max_source_event_dt','target_dropout'}
def make_pipeline(df, model_name):
    features=[c for c in df.columns if c not in EXCLUDE]
    cats=[c for c in features if df[c].dtype=='object' or str(df[c].dtype).startswith(('string','category'))]
    nums=[c for c in features if c not in cats]
    numeric=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())])
    categorical=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    prep=ColumnTransformer([('num',numeric,nums),('cat',categorical,cats)],sparse_threshold=0)
    if model_name=='logistic': clf=LogisticRegression(max_iter=3000,class_weight='balanced',C=0.5,random_state=RANDOM_STATE)
    elif model_name=='random_forest': clf=RandomForestClassifier(n_estimators=400,min_samples_leaf=5,max_features='sqrt',class_weight='balanced_subsample',n_jobs=-1,random_state=RANDOM_STATE)
    elif model_name=='hist_gradient_boosting': clf=HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,max_leaf_nodes=7,l2_regularization=2.0,random_state=RANDOM_STATE)
    return Pipeline([('prep',prep),('model',clf)]),features

metrics=[]; pred_frames=[]
for h in [7,14,30]:
    df=pd.read_csv(FEATURES_OUT/f'rich_snapshot_{h}d.csv')
    df['split']=df.student_id.map(split_map)
    # Exclude examples whose signup-relative cutoff lies after the data extraction end. (Known administrative censoring, not future student behavior.)
    df['snapshot_cutoff']=pd.to_datetime(df.snapshot_cutoff)
    df=df[df.snapshot_cutoff<=observation_end].copy()
    for model_name in ['logistic','random_forest','hist_gradient_boosting']:
        model,features=make_pipeline(df,model_name)
        tr=df[df.split.eq('train')]; va=df[df.split.eq('validation')]; te=df[df.split.eq('test')]
        model.fit(tr[features],tr.target_dropout.astype(int))
        pv=model.predict_proba(va[features])[:,1]; pt=model.predict_proba(te[features])[:,1]
        # Save probabilities for the later cascade notebook.
        for part_name,part,probs in [('validation',va,pv),('test',te,pt)]:
            pred_frames.append(pd.DataFrame({'student_id':part.student_id.to_numpy(),'split':part_name,'horizon_days':h,'model':model_name,'target_dropout':part.target_dropout.to_numpy(),'probability':probs,'snapshot_cutoff':part.snapshot_cutoff.astype(str).to_numpy()}))
        for floor in PRECISION_FLOORS:
            t,ok,val_rec,val_prec=choose_threshold(va.target_dropout.to_numpy(),pv,floor)
            if ok:
                m=metric_row(te.target_dropout.to_numpy(),pt,t)
            else:
                m={k:np.nan for k in ['precision','recall','TP','FP','FN','TN','flagged','flagged_rate']}
                m.update({'PR_AUC':average_precision_score(te.target_dropout,pt),'ROC_AUC':roc_auc_score(te.target_dropout,pt) if te.target_dropout.nunique()==2 else np.nan})
            metrics.append({'horizon_days':h,'model':model_name,'precision_floor':floor,'validation_feasible':ok,'threshold':t,'validation_precision':val_prec,'validation_recall':val_rec,**m})
        with open(MODELS_OUT/f'{h}d_{model_name}.pkl','wb') as f: pickle.dump({'model':model,'features':features},f)

metric_df=pd.DataFrame(metrics); pred_df=pd.concat(pred_frames,ignore_index=True)
display(metric_df.sort_values(['horizon_days','precision_floor','recall'],ascending=[True,True,False]))
metric_df.to_csv(METRICS_OUT/'rich_snapshot_model_comparison.csv',index=False)
pred_df.to_csv(PRED_OUT/'snapshot_validation_test_probabilities.csv',index=False)
print('Saved features, models, thresholds metrics, and validation/test probabilities under',OUT)

,horizon_days,model,precision_floor,validation_feasible,threshold,validation_precision,validation_recall,precision,recall,TP,FP,FN,TN,flagged,flagged_rate,PR_AUC,ROC_AUC
3,7,random_forest,0.4,True,0.296964,0.410714,0.851852,0.366667,0.709677,22,38,9,31,60,0.60,0.427921,0.582515
6,7,hist_gradient_boosting,0.4,True,0.219755,0.400000,0.740741,0.375000,0.677419,21,35,10,34,56,0.56,0.426125,0.579710
0,7,logistic,0.4,True,0.473507,0.411765,0.777778,0.377358,0.645161,20,33,11,36,53,0.53,0.501170,0.601216
1,7,logistic,0.5,True,0.749883,0.516129,0.592593,0.500000,0.483871,15,15,16,54,30,0.30,0.501170,0.601216
4,7,random_forest,0.5,True,0.485869,0.531250,0.629630,0.394737,0.483871,15,23,16,46,38,0.38,0.427921,0.582515
7,7,hist_gradient_boosting,0.5,True,0.741739,0.571429,0.148148,0.571429,0.129032,4,3,27,66,7,0.07,0.426125,0.579710
2,7,logistic,0.6,True,0.907782,0.600000,0.333333,0.625000,0.322581,10,6,21,63,16,0.16,0.501170,0.601216
5,7,random_forest,0.6,True,0.841669,0.666667,0.148148,0.571429,0.129032,4,3,27,66,7,0.07,0.427921,0.582515
8,7,hist_gradient_boosting,0.6,True,0.791869,0.600000,0.111111,0.400000,0.064516,2,3,29,66,5,0.05,0.426125,0.579710
9,14,logistic,0.4,True,0.365948,0.409836,0.925926,0.377049,0.741935,23,38,8,31,61,0.61,0.538153,0.663862


Saved features, models, thresholds metrics, and validation/test probabilities under /Users/abishek/Documents/Projects/student_success_assistant/outputs/rich_snapshot


## 5. Leakage and coverage audit

The assertions below check row uniqueness, the fixed cutoffs, and predictor exclusions. The feature builder filters raw attempts before calculating any aggregate. This is the key no-future-event invariant. Do not use the `target_dropout`, `student_id`, or cutoff metadata as predictors.

In [6]:
audit=[]
for h in [7,14,30]:
    df=pd.read_csv(FEATURES_OUT/f'rich_snapshot_{h}d.csv')
    assert df.student_id.is_unique
    assert df.target_dropout.isin([0,1]).all()
    assert 'refund_requested' not in df.columns and 'explanation_text' not in df.columns
    audit.append({'horizon_days':h,'rows':len(df),'unique_students':df.student_id.nunique(),'positive_rate':df.target_dropout.mean(),'feature_count':len([c for c in df.columns if c not in EXCLUDE]),'snapshot_cutoff_after_observation_end':int((pd.to_datetime(df.snapshot_cutoff)>observation_end).sum())})
audit=pd.DataFrame(audit); display(audit); audit.to_csv(METRICS_OUT/'rich_snapshot_feature_audit.csv',index=False)
print('Caution: threshold feasibility is based on one small validation cohort. Use later-cohort and repeated-cutoff checks before operational use.')

,horizon_days,rows,unique_students,positive_rate,feature_count,snapshot_cutoff_after_observation_end
0,7,500,500,0.25,53,0
1,14,500,500,0.25,53,0
2,30,500,500,0.25,53,0


Caution: threshold feasibility is based on one small validation cohort. Use later-cohort and repeated-cutoff checks before operational use.
